In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import mlflow
import mlflow.keras
from mlflow.models.signature import infer_signature
import os
import warnings
warnings.filterwarnings('ignore')

c:\Users\Asus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load the data set
df = pd.read_csv("../data/filtered_icfes_data_cesar.csv")
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

print(f"Dimensiones: {df.shape}")
print(f"\nColumnas: {list(df.columns)}")

Dimensiones: (63127, 28)

Columnas: ['periodo', 'cole_area_ubicacion', 'cole_bilingue', 'cole_calendario', 'cole_caracter', 'cole_cod_dane_establecimiento', 'cole_cod_dane_sede', 'cole_cod_mcpio_ubicacion', 'cole_jornada', 'cole_naturaleza', 'estu_cod_reside_mcpio', 'estu_genero', 'estu_nacionalidad', 'fami_cuartoshogar', 'fami_educacionmadre', 'fami_educacionpadre', 'fami_estratovivienda', 'fami_personashogar', 'fami_tieneautomovil', 'fami_tienecomputador', 'fami_tieneinternet', 'fami_tienelavadora', 'punt_ingles', 'punt_matematicas', 'punt_sociales_ciudadanas', 'punt_c_naturales', 'punt_lectura_critica', 'punt_global']


In [4]:
df = df.astype({
    "periodo": "float32",
    "cole_area_ubicacion": "category",
    "cole_bilingue": "category",
    "cole_calendario": "category",
    "cole_caracter": "category",
    "cole_cod_dane_establecimiento": "category",
    "cole_cod_dane_sede": "category",
    "cole_cod_mcpio_ubicacion": "category",
    "cole_jornada": "category",
    "cole_naturaleza": "category",
    "estu_cod_reside_mcpio": "category",
    "estu_genero": "category",
    "estu_nacionalidad": "category",
    "fami_cuartoshogar": "category",
    "fami_educacionmadre": "category",
    "fami_educacionpadre": "category",
    "fami_estratovivienda": "category",
    "fami_personashogar": "category",
    "fami_tieneautomovil": "category",
    "fami_tienecomputador": "category",
    "fami_tieneinternet": "category",
    "fami_tienelavadora": "category",
})

# Binarias a bool
binary_map = {
    "cole_area_ubicacion":  {"URBANO": True,  "RURAL": False},
    "cole_bilingue":        {"S": True,        "N": False},
    "cole_calendario":      {"A": True,        "B": False},
    "cole_naturaleza":      {"OFICIAL": True,  "NO OFICIAL": False},
    "estu_genero":          {"M": True,        "F": False},
    "fami_tieneautomovil":  {"Si": True,       "No": False},
    "fami_tienecomputador": {"Si": True,       "No": False},
    "fami_tieneinternet":   {"Si": True,       "No": False},
    "fami_tienelavadora":   {"Si": True,       "No": False},
}
for col, mapping in binary_map.items():
    df[col] = df[col].map(mapping)

df = df.rename(columns={
    "cole_area_ubicacion": "cole_area_urbano",
    "cole_calendario":     "cole_calendario_a",
    "cole_naturaleza":     "cole_oficial",
    "estu_genero":         "estu_masculino",
})

cuartos_map = {
    "Uno": "1", "Dos": "2", "Tres": "3", "Cuatro": "4", "Cinco": "5",
    "Seis": "6+", "Seis o mas": "6+", "Siete": "6+",
    "Ocho": "6+", "Nueve": "6+", "Diez o más": "6+"
}
df["fami_cuartoshogar"] = df["fami_cuartoshogar"].map(cuartos_map)

personas_map = {
    "Una": "1 a 2", "Dos": "1 a 2",
    "Tres": "3 a 4", "Cuatro": "3 a 4",
    "Cinco": "5 a 6", "Seis": "5 a 6",
    "Siete": "7 a 8", "Ocho": "7 a 8",
    "Nueve": "9 o más", "Diez": "9 o más",
    "Once": "9 o más", "Doce o más": "9 o más"
}
df["fami_personashogar"] = df["fami_personashogar"].map(personas_map)

categorical_cols = [
    "cole_caracter", "cole_cod_dane_establecimiento", "cole_cod_dane_sede",
    "cole_cod_mcpio_ubicacion", "cole_jornada", "estu_cod_reside_mcpio",
    "estu_nacionalidad", "fami_cuartoshogar", "fami_educacionmadre",
    "fami_educacionpadre", "fami_estratovivienda", "fami_personashogar",
]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Dimensiones tras preprocesamiento: {df.shape}")
print(f"Valores nulos restantes: {df.isnull().sum().sum()}")

Dimensiones tras preprocesamiento: (63127, 461)
Valores nulos restantes: 0


In [9]:
TARGET_COLS = ['punt_matematicas', 'punt_lectura_critica',
               'punt_c_naturales', 'punt_sociales_ciudadanas', 'punt_ingles']
DROP_COLS   = ['punt_global'] + TARGET_COLS

X = df.drop(columns=DROP_COLS).to_numpy(dtype=np.float32)
y = df[TARGET_COLS].to_numpy(dtype=np.float32)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nX_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")

X shape: (63127, 455)
y shape: (63127, 5)

X_train: (50501, 455) | X_test: (12626, 455)
y_train: (50501, 5) | y_test: (12626, 5)


In [10]:
tracking_dir = os.path.join(os.getcwd(), "mlruns")
mlflow.set_tracking_uri(f"file:///{tracking_dir.replace(os.sep, '/')}")

EXPERIMENT_NAME = "pregunta3_scores"
mlflow.set_experiment(EXPERIMENT_NAME)

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment ID       : {exp.experiment_id}")
print(f"X_train shape       : {X_train.shape}")
print(f"X_test  shape       : {X_test.shape}")
print(f"Targets             : {TARGET_COLS}")

2026/05/24 13:43:06 INFO mlflow.tracking.fluent: Experiment with name 'pregunta3_scores' does not exist. Creating a new experiment.


MLflow tracking URI : file:///c:/Users/Asus/Desktop/Clases/Analítica/Proy2/IIND4130-P2/data_science_3/mlruns
Experiment ID       : 320118271050873575
X_train shape       : (50501, 455)
X_test  shape       : (12626, 455)
Targets             : ['punt_matematicas', 'punt_lectura_critica', 'punt_c_naturales', 'punt_sociales_ciudadanas', 'punt_ingles']


In [12]:
#helper functions
TARGET_COLS = ['punt_matematicas', 'punt_lectura_critica',
               'punt_c_naturales', 'punt_sociales_ciudadanas', 'punt_ingles']

def build_norm_layer(X_train_arr):
    norm = tf.keras.layers.Normalization()
    norm.adapt(X_train_arr)
    return norm

def build_model(input_dim, arch_spec, norm_layer):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = norm_layer(inputs)
    for layer_cfg in arch_spec["layers"]:
        x = tf.keras.layers.Dense(layer_cfg["units"], activation=layer_cfg["activation"])(x)
        if layer_cfg.get("batch_norm", False):
            x = tf.keras.layers.BatchNormalization()(x)
        if layer_cfg.get("dropout_rate", 0.0) > 0:
            x = tf.keras.layers.Dropout(layer_cfg["dropout_rate"])(x)
    #5 salidas, una por área
    outputs = tf.keras.layers.Dense(5, activation="linear")(x)
    model = tf.keras.Model(inputs, outputs)
    return model

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0)
    metrics = {}
    #métricas por materia
    for i, col in enumerate(TARGET_COLS):
        mae  = mean_absolute_error(y_test[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_test[:, i], y_pred[:, i]))
        r2   = r2_score(y_test[:, i], y_pred[:, i])
        metrics[f"mae_{col}"]  = mae
        metrics[f"rmse_{col}"] = rmse
        metrics[f"r2_{col}"]   = r2
    #métricas globales
    metrics["mae_mean"]  = np.mean([metrics[f"mae_{c}"]  for c in TARGET_COLS])
    metrics["rmse_mean"] = np.mean([metrics[f"rmse_{c}"] for c in TARGET_COLS])
    metrics["r2_mean"]   = np.mean([metrics[f"r2_{c}"]   for c in TARGET_COLS])
    return metrics, y_pred

def log_mlflow_run(run_name, arch_spec, params, Xtr, ytr, Xte, yte, feature_set_name="full"):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("architecture",   run_name)
        mlflow.log_param("feature_set",    feature_set_name)
        mlflow.log_param("epochs_max",     params["epochs"])
        mlflow.log_param("batch_size",     params["batch_size"])
        mlflow.log_param("learning_rate",  params["lr"])
        mlflow.log_param("n_features",     Xtr.shape[1])

        norm  = build_norm_layer(Xtr)
        model = build_model(Xtr.shape[1], arch_spec, norm)
        model.compile(
            loss="mse",
            optimizer=tf.keras.optimizers.Adam(learning_rate=params["lr"]),
            metrics=["mae"]
        )
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=10, restore_best_weights=True
            )
        ]
        history = model.fit(
            Xtr, ytr,
            validation_split=0.2,
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            callbacks=callbacks,
            verbose=0
        )
        epochs_run = len(history.history["loss"])
        metrics, _ = evaluate_model(model, Xte, yte)

        mlflow.log_metric("epochs_run", epochs_run)
        for k, v in metrics.items():
            mlflow.log_metric(k, v)

        print(f"  [{run_name}] epochs={epochs_run:3d} | "
              f"MAE={metrics['mae_mean']:.4f} | "
              f"RMSE={metrics['rmse_mean']:.4f} | "
              f"R²={metrics['r2_mean']:.4f}")
    return metrics, model

In [13]:
#definición de diferentes arquitecturas a probar
ARCHITECTURES = {
    "arch_1": {
        "layers": [
            {"units": 256, "activation": "relu", "batch_norm": False, "dropout_rate": 0.0},
            {"units": 128, "activation": "relu", "batch_norm": False, "dropout_rate": 0.0},
        ]
    },
    "arch_2": {
        "layers": [
            {"units": 256, "activation": "relu", "batch_norm": False, "dropout_rate": 0.3},
            {"units": 128, "activation": "relu", "batch_norm": False, "dropout_rate": 0.3},
            {"units": 64,  "activation": "relu", "batch_norm": False, "dropout_rate": 0.2},
        ]
    },
    "arch_3": {
        "layers": [
            {"units": 512, "activation": "relu", "batch_norm": False, "dropout_rate": 0.0},
            {"units": 256, "activation": "relu", "batch_norm": False, "dropout_rate": 0.0},
            {"units": 128, "activation": "relu", "batch_norm": False, "dropout_rate": 0.0},
            {"units": 64,  "activation": "relu", "batch_norm": False, "dropout_rate": 0.0},
        ]
    },
    "arch_4": {
        "layers": [
            {"units": 256, "activation": "relu", "batch_norm": True,  "dropout_rate": 0.0},
            {"units": 128, "activation": "relu", "batch_norm": True,  "dropout_rate": 0.0},
            {"units": 64,  "activation": "relu", "batch_norm": True,  "dropout_rate": 0.0},
        ]
    },
    "arch_5": {
        "layers": [
            {"units": 512, "activation": "relu", "batch_norm": False, "dropout_rate": 0.3},
            {"units": 256, "activation": "relu", "batch_norm": False, "dropout_rate": 0.2},
            {"units": 128, "activation": "relu", "batch_norm": False, "dropout_rate": 0.2},
            {"units": 64,  "activation": "relu", "batch_norm": False, "dropout_rate": 0.1},
        ]
    },
}

TRAINING_CONFIGS = {
    "adam_1e3":    {"lr": 1e-3, "epochs": 100, "batch_size": 32},
    "adam_5e4":    {"lr": 5e-4, "epochs": 100, "batch_size": 32},
    "adam_1e4":    {"lr": 1e-4, "epochs": 100, "batch_size": 32},
    "large_batch": {"lr": 1e-3, "epochs": 100, "batch_size": 128},
}

print(f"Arquitecturas: {list(ARCHITECTURES.keys())}")
print(f"Configs: {list(TRAINING_CONFIGS.keys())}")

Arquitecturas: ['arch_1', 'arch_2', 'arch_3', 'arch_4', 'arch_5']
Configs: ['adam_1e3', 'adam_5e4', 'adam_1e4', 'large_batch']


In [14]:
#Feature sets
dane_cols = [c for c in df.drop(columns=['punt_global'] + TARGET_COLS).columns
             if c.startswith("cole_cod_dane_establecimiento_")
             or c.startswith("cole_cod_dane_sede_")]

X_no_dane = df.drop(columns=['punt_global'] + TARGET_COLS + dane_cols).to_numpy(dtype=np.float32)

contextual_cols = [c for c in df.columns if any(c.startswith(p) for p in [
    "cole_area", "cole_bilingue", "cole_calendario", "cole_oficial",
    "cole_jornada", "cole_caracter", "fami_", "estu_masculino"
])]
X_contextual = df[contextual_cols].to_numpy(dtype=np.float32)

X_no_dane_tr, X_no_dane_te, _, _ = train_test_split(X_no_dane, y, test_size=0.2, random_state=42)
X_ctx_tr, X_ctx_te, _, _         = train_test_split(X_contextual, y, test_size=0.2, random_state=42)

FEATURE_SETS = {
    "full_461":   (X_train,      X_test,      y_train, y_test),
    "no_dane":    (X_no_dane_tr, X_no_dane_te, y_train, y_test),
    "contextual": (X_ctx_tr,     X_ctx_te,     y_train, y_test),
}

for name, (Xtr, Xte, _, _) in FEATURE_SETS.items():
    print(f"  Feature set '{name}': {Xtr.shape[1]} features")

  Feature set 'full_461': 455 features
  Feature set 'no_dane': 105 features
  Feature set 'contextual': 53 features


In [15]:
print("=== Fase 1: Búsqueda de arquitectura (full_461, adam_1e3) ===\n")

Xtr_f, Xte_f, ytr_f, yte_f = FEATURE_SETS["full_461"]
base_cfg = TRAINING_CONFIGS["adam_1e3"]

phase1_results = {}
for arch_name, arch_spec in ARCHITECTURES.items():
    metrics, model = log_mlflow_run(
        run_name=f"p1_{arch_name}",
        arch_spec=arch_spec,
        params=base_cfg,
        Xtr=Xtr_f, ytr=ytr_f,
        Xte=Xte_f, yte=yte_f,
        feature_set_name="full_461"
    )
    phase1_results[arch_name] = metrics

# Mejor arquitectura por R² medio
best_arch_name = max(phase1_results, key=lambda k: phase1_results[k]["r2_mean"])
print(f"\n--- Ranking Fase 1 (por R² medio) ---")
for arch, m in sorted(phase1_results.items(), key=lambda x: -x[1]["r2_mean"]):
    print(f"  {arch:25s} | MAE={m['mae_mean']:.4f} | R²={m['r2_mean']:.4f}")
print(f"\nMejor arquitectura: {best_arch_name}")

=== Fase 1: Búsqueda de arquitectura (full_461, adam_1e3) ===

  [p1_arch_1] epochs= 25 | MAE=6.9795 | RMSE=8.8028 | R²=0.2715
  [p1_arch_2] epochs= 34 | MAE=6.9035 | RMSE=8.7061 | R²=0.2878
  [p1_arch_3] epochs= 25 | MAE=6.9592 | RMSE=8.7876 | R²=0.2743
  [p1_arch_4] epochs= 23 | MAE=6.8677 | RMSE=8.6896 | R²=0.2905
  [p1_arch_5] epochs= 40 | MAE=6.9384 | RMSE=8.7605 | R²=0.2790

--- Ranking Fase 1 (por R² medio) ---
  arch_4                    | MAE=6.8677 | R²=0.2905
  arch_2                    | MAE=6.9035 | R²=0.2878
  arch_5                    | MAE=6.9384 | R²=0.2790
  arch_3                    | MAE=6.9592 | R²=0.2743
  arch_1                    | MAE=6.9795 | R²=0.2715

Mejor arquitectura: arch_4


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.size"] = 14
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 12

arch_names = list(phase1_results.keys())
mae_values = [phase1_results[a]["mae_mean"] for a in arch_names]
best_idx   = mae_values.index(min(mae_values))

colors = ["#e07b00" if i == best_idx else "#1a3a6b" for i in range(len(arch_names))]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(arch_names, mae_values, color=colors, edgecolor="white", linewidth=0.5)

for bar, val in zip(bars, mae_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_xlabel("Arquitectura")
ax.set_ylabel("MAE Promedio")
ax.set_title("Comparacion de Arquitecturas")
ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#e07b00", label="Mejor arquitectura"),
                   Patch(facecolor="#1a3a6b", label="Otras arquitecturas")]
ax.legend(handles=legend_elements)

plt.tight_layout()
plt.savefig("grafico_q3_arquitecturas.png", dpi=300, bbox_inches="tight", transparent=True, facecolor="white")
plt.show()
print(f"Mejor: {arch_names[best_idx]}  MAE={mae_values[best_idx]:.4f}")
print("Guardado: grafico_q3_arquitecturas.png")